##  Installations


In [1]:
pip install gurobipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB



In [4]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
df = pd.read_excel("iett_veri.xlsx")
df.columns = df.columns.str.strip()

df["Hat Kodu"] = df["Hat Kodu"].astype(str).str.strip()
df["Hat Başı"] = df["Hat Başı"].astype(str).str.strip()
df["Hat Sonu"] = df["Hat Sonu"].astype(str).str.strip()

## Sets



In [6]:
#Sets
H = df["Hat Kodu"].unique().tolist()
B = pd.concat([
    df["Hat Başı"],
    df["Hat Sonu"]
]).dropna().astype(str).str.strip().unique().tolist()
Y = ["Elektrik", "Hidrojen"]

## Parameters

In [7]:
L = dict(zip(df["Hat Kodu"], df["Güzergah Uzunluğu"] / 1000))  # METRE -> KM
S = dict(zip(df["Hat Kodu"], df["Günlük Sefer Sayısı"]))
A = dict(zip(df["Hat Kodu"], df["Araç Sayısı"]))
D = {h: S[h] * L[h] for h in H}  

In [8]:
coord_cols = [
    "Hatbaşı Boylam",
    "Hatbaşı Enlem",
    "Hatsonu Boylam",
    "Hatsonu Enlem"
]

for col in coord_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    

In [9]:
B_coord = {}

for _, row in df.iterrows():
    bas = row["Hat Başı"]
    son = row["Hat Sonu"]

    # İlk sıraya Enlem, ikinci sıraya Boylam gelecek şekilde düz kaydediyoruz:
    B_coord[bas] = (row["Hatbaşı Enlem"], row["Hatbaşı Boylam"])
    B_coord[son] = (row["Hatsonu Enlem"], row["Hatsonu Boylam"])

In [10]:
# Haversine fonksiyonu
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):

    R = 6371  # km yarıçap

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2)**2
        + cos(lat1)
        * cos(lat2)
        * sin(dlon / 2)**2
    )

    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c


In [11]:
d = {}

for _, row in df.iterrows():
    h = row["Hat Kodu"]

    start_lat = row["Hatbaşı Enlem"]  # <── Enlem değişkenine Boylam sütununu eşitledik
    start_lon = row["Hatbaşı Boylam"]   # <── Boylam değişkenine Enlem sütununu eşitledik

    end_lat   = row["Hatsonu Enlem"]   # <── Enlem değişkenine Boylam sütununu eşitledik
    end_lon   = row["Hatsonu Boylam"]    # <── Boylam değişkenine Enlem sütununu eşitledik

    # GERİYE KALAN ALTTAKİ 'FOR B IN B:' DÖNGÜSÜ AYNEREN KALACAK, DEĞİŞTİRME!
    for b in B:
        b_lat, b_lon = B_coord[b]
        dist_start = haversine(start_lat, start_lon, b_lat, b_lon)
        dist_end = haversine(end_lat, end_lon, b_lat, b_lon)
        d[h, b] = min(dist_start, dist_end)

## Yakıt Türüne Bağlı Parametreler

In [13]:

#R_Y: Y yakıtının menzil mesafesi
R = {
    "Elektrik": 500,   # km
    "Hidrojen": 500   # km
}

#T_op: Günlük operasyon süresi
T_op = 24 * 60          # dakika/gün


#T_Y: Y yakıt türünün şarj süresi
T = {
    "Elektrik": 210,    # dakika, 3.5 saat
    "Hidrojen": 7       # dakika
}

#Y yakıt istasyonu kurma maliyeti
C_open = {
    "Elektrik": 284_000/365,
    "Hidrojen": 2_950_000/365
}

#c_Y: y yakıt türü charger/ dispenser maliyeti
C_unit = {
    "Elektrik": 144_795/365,
    "Hidrojen": 0
}

#g_Y: y yakıt türü km başına yakıt/enerji maliyeti
C_km = {
    "Elektrik": 0.42,
    "Hidrojen": 7.5
}

#E_Y: Y yakıt türünün km başına emisyonu
E_km = {
    "Elektrik": 333.55,
    "Hidrojen": 70.131
}


# U[h, y]: h hattında çalışan bir aracın y yakıt türü altında
# günlük ortalama şarj/dolum zaman yükü
# Birim: dakika/gün
U = {
    (h, y): (D[h] / (R[y] * A[h])) * T[y]
    for h in H
    for y in Y
}

V_H   = 0.075     # kg/km  (37,5 kg tank / 500 km menzil — KARSAN e-ATA Hydrogen)
V_E   = 1.2       # kwh/km
Cap_H = 1000      # kg/gün  istasyon dolum kapasitesi
Cap_E = 180*24

In [14]:
model = gp.Model("Line&Fuel Optimization Model")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2817913
Academic license 2817913 - for non-commercial use only - registered to ze___@std.yildiz.edu.tr


## Karar Değişkenleri

In [15]:
# X_BY
X = model.addVars(
    B,
    Y,
    vtype=GRB.BINARY,
    name="X"
)

In [16]:
# K_BY: B noktasındaki Y yakıt soket/pompa sayısı
K = model.addVars(
    B,
    Y,
    vtype=GRB.INTEGER,
    lb=0,
    name="K"
)

In [17]:
# J_HBY
J = model.addVars(
    H,
    B,
    Y,
    vtype=GRB.BINARY,
    name="J"
)

In [18]:
# Z_HY
Z = model.addVars(
    H,
    Y,
    vtype=GRB.BINARY,
    name="Z"
)

In [ ]:
#Kontrol
print("Hat sayısı:", len(H))
print("Aday istasyon sayısı:", len(B))
print("Yakıt türü sayısı:", len(Y))

print("X değişken sayısı:", len(X))
print("K değişken sayısı:", len(K))
print("J değişken sayısı:", len(J))
print("Z değişken sayısı:", len(Z))

Hat sayısı: 434
Aday istasyon sayısı: 246
Yakıt türü sayısı: 2
X değişken sayısı: 492
K değişken sayısı: 492
J değişken sayısı: 213528
Z değişken sayısı: 868


In [20]:
# Şarj cihazı sayısı (her cihazda 2 soket)
N = model.addVars(
    B,
    vtype=GRB.INTEGER,
    lb=0,
    name="N"
)

# K[b, "Elektrik"] her zaman 2'nin katı olmalı
for b in B:
    model.addConstr(
        K[b, "Elektrik"] == 2 * N[b],
        name=f"Soket_Cift_{b}"
    )

## Kısıtlar

In [21]:
#Kısıtlar
# Her hat yalnızca bir yakıt türü seçebilir
for h in H:
    model.addConstr(
        gp.quicksum(Z[h, y] for y in Y) == 1,
        name=f"Tek_Yakit_Secimi_{h}"
    )

In [22]:
# 2. Hat, seçtiği yakıt türünde tam bir istasyona atanır
for h in H:
    for y in Y:
        model.addConstr(
            gp.quicksum(J[h, b, y] for b in B) == Z[h, y],
            name=f"Istasyon_Atama_{h}_{y}"
        )

In [23]:
# 3. Açılmayan istasyona hat atanamaz
for h in H:
    for b in B:
        for y in Y:
            model.addConstr(
                J[h, b, y] <= X[b, y],
                name=f"Atama_Icin_Istasyon_{h}_{b}_{y}"
            )

In [24]:
for b in B:
    for y in Y:
        model.addConstr(
            X[b, y] <= gp.quicksum(J[h, b, y] for h in H),
            name=f"Kullanilmayan_Istasyon_Acilmasin_{b}_{y}"
        )

In [25]:
# 4. Soket/pompa açmak için istasyon kurulmalıdır
M = len(H) * max(A.values())

for b in B:
    model.addConstr(K[b, "Elektrik"] <= M * X[b, "Elektrik"], name=f"Soket_Istasyon_Baglantisi_{b}")



In [26]:
# 5. İstasyon kurulursa en az bir soket/pompa olmalıdır
for b in B:
    model.addConstr(K[b, "Elektrik"] >= 2*X[b, "Elektrik"], name=f"Min_Soket_{b}")

In [27]:
# 6. İstasyon kapasitesi aşılmamalıdır
# Toplam günlük talep = A[h] * N[h, y]
# Bu ifade D[h] / R[y] ile aynı anlama gelir.

for b in B:
    model.addConstr(
        gp.quicksum(D[h] * V_E * J[h, b, "Elektrik"] for h in H)
        <= Cap_E * K[b, "Elektrik"],
        name=f"Kapasite_Elektrik_{b}")
    
    # Hidrojen: kütle-tabanlı istasyon kapasitesi (kg/gün)
    model.addConstr(
        gp.quicksum(D[h] * V_H * J[h, b, "Hidrojen"] for h in H)
        <= Cap_H * X[b, "Hidrojen"],
        name=f"Kapasite_Hidrojen_{b}")

In [28]:
#Bir noktada en fazla 1 hidrojen açılabilir
for b in B:
    model.addConstr(
        K[b, "Hidrojen"] == X[b, "Hidrojen"]
    )

In [29]:
# İstanbul genelinde kurulabilecek maksimum toplam hidrojen istasyon sayısı
model.addConstr(
    gp.quicksum(X[b, "Hidrojen"] for b in B) <= 20, 
    name="Maks_Toplam_H2_Istasyon_Siniri"
)

<gurobi.Constr *Awaiting Model Update*>

## Amaç Fonksiyonu

Amaç fonksiyonu; kurulum maliyeti, operasyonel maliyet, emisyon ve erişim mesafesini birlikte minimize eder. AHP ile belirlenen ağırlıklar burada kullanılabilir.

In [ ]:
# AHP veya senaryo ağırlıkları

w_cost = 0.185
w_emission = 0.214
w_access = 0.601


scale_cost     = 1_000_000      
scale_emission = 10_000_000     
scale_access   = 250_000


fixed_station_cost = gp.quicksum(
    C_open[y] * X[b, y]
    for b in B
    for y in Y
)

socket_pump_cost = gp.quicksum(
    C_unit["Elektrik"] * K[b, "Elektrik"]
    for b in B
)

operation_cost = gp.quicksum(
    C_km[y] * D[h] * Z[h, y]
    for h in H
    for y in Y
)

emission_cost = gp.quicksum(
    E_km[y] * D[h] * Z[h, y]
    for h in H
    for y in Y
)

access_distance = gp.quicksum(
    2 * d[h, b] * (D[h] / R[y]) * J[h, b, y]
    for h in H
    for b in B
    for y in Y
)
total_cost = fixed_station_cost + socket_pump_cost + operation_cost

model.setObjective(
    w_cost     * (total_cost / scale_cost) +
    w_emission * (emission_cost / scale_emission) +
    w_access   * (access_distance / scale_access),
    GRB.MINIMIZE
)


In [31]:
# ── GÜNCEL BİLEŞEN ETKİ VE YÜZDELİK ANALİZ KODU ──

if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    # 1. Ölçeklendirilmiş (Scaled) Değerleri Hesaplayalım
    scaled_cost = total_cost.getValue() / scale_cost
    scaled_emis = emission_cost.getValue() / scale_emission
    scaled_acc  = access_distance.getValue() / scale_access

    # 2. Ağırlıklandırılmış Nihai Katkıları Hesaplayalım
    weighted_cost = w_cost * scaled_cost
    weighted_emis = w_emission * scaled_emis
    weighted_acc  = w_access * scaled_acc

    # Toplam Skor (Amaç Fonksiyonu Değeri)
    total_score = weighted_cost + weighted_emis + weighted_acc

    # 3. Sonuçları Ekrana Düzenli Yazdıralım
    print("="*60)
    print(f"   GÜNCEL MODEL BİLEŞEN ANALİZİ (Obj: {model.ObjVal:.4f})")
    print("="*60)
    print(f"Maliyet Katkısı : {w_cost:.3f} × {scaled_cost:.4f} = {weighted_cost:.4f}")
    print(f"Emisyon Katkısı : {w_emission:.3f} × {scaled_emis:.4f} = {weighted_emis:.4f}")
    print(f"Erişim Katkısı  : {w_access:.3f} × {scaled_acc:.4f} = {weighted_acc:.4f}")
    print("-"*60)
    
    # 4. Yüzdesel Etki Dağılımı (Modeli hangisi yönetiyor?)
    if total_score > 0:
        print("BİLEŞENLERİN TOPLAM ETKİ İÇİNDEKİ YÜZDESEL PAYLARI:")
        print(f" • Maliyet Etkisi  : %{(weighted_cost / total_score) * 100:.2f}")
        print(f" • Emisyon Etkisi  : %{(weighted_emis / total_score) * 100:.2f}")
        print(f" • Erişim Etkisi   : %{(weighted_acc / total_score) * 100:.2f}")
    print("="*60)
else:
    print("Model henüz çözülmediği için bileşen katkıları hesaplanamıyor.")

Model henüz çözülmediği için bileşen katkıları hesaplanamıyor.


## Model Çözümü

In [32]:
model.Params.TimeLimit = 300
model.Params.MIPGap = 0.01

model.optimize()


Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 255H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.01

Academic license 2817913 - for non-commercial use only - registered to ze___@std.yildiz.edu.tr
Optimize a model with 216799 rows, 215626 columns and 1072574 nonzeros (Min)
Model fingerprint: 0x625f05f2
Model has 213470 linear objective coefficients
Variable types: 0 continuous, 215626 integer (214888 binary)
Coefficient statistics:
  Matrix range     [2e-01, 3e+04]
  Objective range  [7e-09, 6e-01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Presolve removed 2590 rows and 1606 columns (presolve time = 5s)...
Presolve removed 2590 rows and 1606 columns
Presolve time: 7.66s
Presolved: 214209 rows, 214020 c

## Sonuçların Tablo Halinde Alınması

In [33]:
print("Fixed station cost:", fixed_station_cost.getValue())
print("Socket/pump cost:", socket_pump_cost.getValue())
print("Operation cost:", operation_cost.getValue())
print("Total cost:", (fixed_station_cost + socket_pump_cost + operation_cost).getValue())
print("Emission:", emission_cost.getValue())
print("Access distance:", access_distance.getValue())

Fixed station cost: 173315.06849315064
Socket/pump cost: 32529.28767123288
Operation cost: 2098503.873910802
Total cost: 2304348.2300751847
Emission: 98680902.55784526
Access distance: 1303.1371599770177


In [34]:
# ── Hat-Yakıt-İstasyon Atama Tablosu ────────────────────────────────────────
arows = []

# Modelin başarılı bir şekilde çözüldüğünden emin olmak için kontrol
if model.Status == GRB.OPTIMAL or model.Status == GRB.TIME_LIMIT:
    for h in H:
        for b in B:
            for y in Y:
                # J değişkeninin optimize edilmiş değerini kontrol ediyoruz
                if J[h, b, y].X > 0.5:
                    gunluk_enerji = D[h] * V_E if y == "Elektrik" else 0
                    gunluk_h2    = D[h] * V_H  if y == "Hidrojen" else 0
                    
                    arows.append({
                        "Hat Kodu"          : h,
                        "Yakıt"             : y,
                        "Atandığı İstasyon" : b,
                        "Erişim (km)"       : round(d[h, b], 3),
                        "Günlük Hat-km"     : round(D[h], 1),
                        "Araç Sayısı"       : A[h],
                        "Araç Şarj (dk/gün)": round(U[h, y], 1),
                        "Günlük Enerji (kWh)": round(gunluk_enerji, 1),
                        "Günlük H2 (kg)"    : round(gunluk_h2, 2),
                    })

    assignment_df = pd.DataFrame(arows).sort_values(["Yakıt", "Hat Kodu"])
    print(f"Toplam atanan hat: {len(assignment_df)}")
    
    # Notebook üzerinde daha şık görünmesi için display kullanabilirsin
    display(assignment_df.head(10))
else:
    print("Model henüz optimal bir çözüme ulaşmadı veya çözülemedi. Lütfen önce modeli optimize edin.")

Toplam atanan hat: 434


,Hat Kodu,Yakıt,Atandığı İstasyon,Erişim (km),Günlük Hat-km,Araç Sayısı,Araç Şarj (dk/gün),Günlük Enerji (kWh),Günlük H2 (kg)
3,142,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,3372.1,26,54.5,4046.5,0.0
4,142A,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,1170.7,5,98.3,1404.9,0.0
7,142ES,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,685.0,4,71.9,822.0,0.0
8,142F,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,2598.2,23,47.4,3117.8,0.0
9,142K,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,449.2,3,62.9,539.1,0.0
12,144,Elektrik,HARAMİDERE,0.000,386.6,2,81.2,463.9,0.0
13,144A,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,1167.5,6,81.7,1401.0,0.0
14,144B,Elektrik,KANUNİ SULTAN SÜLEYMAN,10.625,600.9,2,126.2,721.0,0.0
15,144H,Elektrik,KANUNİ SULTAN SÜLEYMAN,9.119,276.0,1,115.9,331.2,0.0
18,145,Elektrik,HARAMİDERE,2.941,2024.5,10,85.0,2429.4,0.0


In [35]:
# ── İSTASYON BAZLI DETAYLI ÖZET KODU ──
station_rows = []

if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    for b in B:
        # İstasyonun açılıp açılmadığını kontrol edelim
        elec_open = X[b, "Elektrik"].X > 0.5
        hydro_open = X[b, "Hidrojen"].X > 0.5
        
        # Eğer bu noktada en az bir istasyon türü açıldıysa detayları çekelim
        if elec_open or hydro_open:
            # O istasyona atanan toplam hat sayısı
            assigned_lines_count = sum(1 for h in H if (J[h, b, "Elektrik"].X > 0.5 or J[h, b, "Hidrojen"].X > 0.5))
            
            # Elektrik soket sayısı (K değişkeninden)
            elec_sockets = int(round(K[b, "Elektrik"].X)) if elec_open else 0
            
            # Hidrojen istasyon/pompa sayısı
            hydro_pumps = int(round(K[b, "Hidrojen"].X)) if hydro_open else 0
            
            station_rows.append({
                "İstasyon Noktası": b,
                "Elektrik İstasyonu": "AÇIK" if elec_open else "Kapalı",
                "Toplam Elektrik Soketi": elec_sockets,
                "Hidrojen İstasyonu": "AÇIK" if hydro_open else "Kapalı",
                "Toplam Hidrojen Pompası": hydro_pumps,
                "Atanan Toplam Hat Sayısı": assigned_lines_count
            })

    stations_summary_df = pd.DataFrame(station_rows).sort_values("Atanan Toplam Hat Sayısı", ascending=False)
    print(f"Toplam Açılan Aktif İstasyon Lokasyonu Sayısı: {len(stations_summary_df)}")
    display(stations_summary_df)
else:
    print("Model henüz çözülmedi veya uygun çözüm bulunamadı.")
    

Toplam Açılan Aktif İstasyon Lokasyonu Sayısı: 30


,İstasyon Noktası,Elektrik İstasyonu,Toplam Elektrik Soketi,Hidrojen İstasyonu,Toplam Hidrojen Pompası,Atanan Toplam Hat Sayısı
20,MECİDİYEKÖY METROBÜS,AÇIK,6,AÇIK,1,43
11,ALİBEYKÖY METRO,AÇIK,10,Kapalı,0,32
23,GÜMÜŞSUYU PERON,AÇIK,10,Kapalı,0,27
19,EMİNÖNÜ,AÇIK,12,Kapalı,0,26
2,HACIOSMAN METRO,AÇIK,2,AÇIK,1,25
0,KANUNİ SULTAN SÜLEYMAN,AÇIK,4,AÇIK,1,21
12,YENİBOSNA METRO,Kapalı,0,AÇIK,1,19
21,İ.Ü. CERRAHPAŞA AVC. KAMP.,AÇIK,14,Kapalı,0,17
4,GİYİMKENT,Kapalı,0,AÇIK,1,16
27,AKSARAY RİNG,Kapalı,0,AÇIK,1,16


In [36]:
# ── DETAYLI İSTASYON DURUMU VE DOLULUK ORANLARI ÖZET KODU ──
station_rows = []

if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    for b in B:
        elec_open = X[b, "Elektrik"].X > 0.5
        hydro_open = X[b, "Hidrojen"].X > 0.5
        
        if elec_open or hydro_open:
            # 1. İstasyon Türünü Belirleme (Tek Sütun)
            if elec_open and hydro_open:
                station_type = "Hibrit (E+H)"
            elif elec_open:
                station_type = "Elektrik"
            else:
                station_type = "Hidrojen"
                
            # 2. Atanan Hat Sayısı
            assigned_lines_count = sum(1 for h in H if (J[h, b, "Elektrik"].X > 0.5 or J[h, b, "Hidrojen"].X > 0.5))
            
            # 3. Soket / Pompa Sayıları
            elec_sockets = int(round(K[b, "Elektrik"].X)) if elec_open else 0
            hydro_pumps = int(round(K[b, "Hidrojen"].X)) if hydro_open else 0
            
            # 4. Kapasite Kullanım Oranı Hesaplama
            # Elektrik Doluluk Oranı = Toplam Talep (kWh) / (Cap_E * Soket Sayısı)
            if elec_open and elec_sockets > 0:
                total_elec_demand = sum(D[h] * V_E * J[h, b, "Elektrik"].X for h in H)
                elec_utilization = (total_elec_demand / (Cap_E * elec_sockets)) * 100
            else:
                elec_utilization = 0.0
                
            # Hidrojen Doluluk Oranı = Toplam Talep (kg) / Cap_H
            if hydro_open:
                total_hydro_demand = sum(D[h] * V_H * J[h, b, "Hidrojen"].X for h in H)
                hydro_utilization = (total_hydro_demand / Cap_H) * 100
            else:
                hydro_utilization = 0.0
            
            # Tabloda tek bir doluluk oranı göstermek için hibrit peronlarda ortalama veya uygun olanı yazdırıyoruz
            if station_type == "Hibrit (E+H)":
                utilization_text = f"E: %{elec_utilization:.1f} / H: %{hydro_utilization:.1f}"
            elif station_type == "Elektrik":
                utilization_text = f"%{elec_utilization:.1f}"
            else:
                utilization_text = f"%{hydro_utilization:.1f}"
                
            station_rows.append({
                "İstasyon Noktası"        : b,
                "İstasyon Türü"           : station_type,
                "Elk. Soket Sayısı"       : elec_sockets,
                "H2 Pompa Sayısı"         : hydro_pumps,
                "Atanan Hat Sayısı"       : assigned_lines_count,
                "Kapasite Kullanım Oranı (%)": utilization_text
            })

    stations_summary_df = pd.DataFrame(station_rows).sort_values("Atanan Hat Sayısı", ascending=False)
    print(f"Toplam Açılan Aktif İstasyon Lokasyonu: {len(stations_summary_df)}")
    display(stations_summary_df)
else:
    print("Model henüz çözülmedi veya uygun çözüm bulunamadı.")

Toplam Açılan Aktif İstasyon Lokasyonu: 30


,İstasyon Noktası,İstasyon Türü,Elk. Soket Sayısı,H2 Pompa Sayısı,Atanan Hat Sayısı,Kapasite Kullanım Oranı (%)
20,MECİDİYEKÖY METROBÜS,Hibrit (E+H),6,1,43,E: %98.9 / H: %99.8
11,ALİBEYKÖY METRO,Elektrik,10,0,32,%93.1
23,GÜMÜŞSUYU PERON,Elektrik,10,0,27,%89.8
19,EMİNÖNÜ,Elektrik,12,0,26,%99.7
2,HACIOSMAN METRO,Hibrit (E+H),2,1,25,E: %15.5 / H: %99.9
0,KANUNİ SULTAN SÜLEYMAN,Hibrit (E+H),4,1,21,E: %20.5 / H: %99.9
12,YENİBOSNA METRO,Hidrojen,0,1,19,%99.9
21,İ.Ü. CERRAHPAŞA AVC. KAMP.,Elektrik,14,0,17,%95.7
4,GİYİMKENT,Hidrojen,0,1,16,%99.8
27,AKSARAY RİNG,Hidrojen,0,1,16,%99.8


In [37]:
# ── GENEL MODEL METRİKLERİ ÖZET KODU ──

if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    # 1. Yakıt türlerine göre atanan hat sayıları
    n_E = sum(1 for h in H if Z[h, "Elektrik"].X > 0.5)
    n_H = sum(1 for h in H if Z[h, "Hidrojen"].X > 0.5)
    
    # 2. Açılan aktif istasyon lokasyon sayıları
    ist_E = sum(1 for b in B if X[b, "Elektrik"].X > 0.5)
    ist_H = sum(1 for b in B if X[b, "Hidrojen"].X > 0.5)
    
    # 3. Toplam kurulan soket ve pompa sayıları
    total_sockets = int(round(sum(K[b, "Elektrik"].X for b in B)))
    total_pumps = int(round(sum(K[b, "Hidrojen"].X for b in B))) # Mantığına göre X'e eşit ama kontrol için K'dan çekiyoruz
    
    # Raporlama için DataFrame oluşturma
    macro_summary = pd.DataFrame([{
        "Toplam Hat Sayısı"              : len(H),
        "Elektrikli Hat Sayısı"          : n_E,
        "Hidrojenli Hat Sayısı"          : n_H,
        "Elektrikli Hat Oranı (%)"       : round(100 * n_E / len(H), 2),
        "Hidrojenli Hat Oranı (%)"       : round(100 * n_H / len(H), 2),
        "Açılan Elektrik İstasyon Sayısı" : ist_E,
        "Açılan Hidrojen İstasyon Sayısı" : ist_H,
        "Toplam Kurulan Elektrik Soketi"  : total_sockets,
        "Toplam Kurulan Hidrojen Pompası" : total_pumps,
    }])
    
    print("="*50)
    print("        YENİ MODEL GENEL METRİKLER ÖZETİ")
    print("="*50)
    display(macro_summary.T.rename(columns={0: "Değer"}))
    print("="*50)
else:
    print("Model optimize edilemedi veya uygun bir çözüm bulunamadı. Lütfen kısıtları ve optimize hücesini kontrol edin.")

        YENİ MODEL GENEL METRİKLER ÖZETİ


,Değer
Toplam Hat Sayısı,434.00
Elektrikli Hat Sayısı,181.00
Hidrojenli Hat Sayısı,253.00
Elektrikli Hat Oranı (%),41.71
Hidrojenli Hat Oranı (%),58.29
Açılan Elektrik İstasyon Sayısı,15.00
Açılan Hidrojen İstasyon Sayısı,20.00
Toplam Kurulan Elektrik Soketi,82.00
Toplam Kurulan Hidrojen Pompası,20.00


In [38]:
# ── 1. HAT BAZLI DETAYLI ATAMA TABLOSU ──
hat_rows = []

if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    for h in H:
        for b in B:
            for y in Y:
                if J[h, b, y].X > 0.5:
                    tuketim = D[h] * V_E if y == "Elektrik" else D[h] * V_H
                    birim = "kWh" if y == "Elektrik" else "kg"
                    
                    hat_rows.append({
                        "Hat Kodu": h,
                        "Seçilen Yakıt": y,
                        "Atandığı İstasyon Peronu": b,
                        "İstasyona Erişim Mesafesi (km)": round(d[h, b], 3),
                        "Günlük Toplam Hat-km": round(D[h], 1),
                        "Araç Sayısı": A[h],
                        "Günlük Şarj/Dolum Yükü (dk)": round(U[h, y], 1),
                        "Günlük Yakıt Tüketimi": f"{round(tuketim, 2)} {birim}"
                    })
                    
    hat_df = pd.DataFrame(hat_rows).sort_values(["Seçilen Yakıt", "Hat Kodu"])
    print(f"✔ Toplam Atanan Hat Sayısı: {len(hat_df)}")
    display(hat_df.head(20)) # İlk 20 satırı gösterir, hepsini görmek için .head() kısmını silebilirsin.

✔ Toplam Atanan Hat Sayısı: 434


,Hat Kodu,Seçilen Yakıt,Atandığı İstasyon Peronu,İstasyona Erişim Mesafesi (km),Günlük Toplam Hat-km,Araç Sayısı,Günlük Şarj/Dolum Yükü (dk),Günlük Yakıt Tüketimi
3,142,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,3372.1,26,54.5,4046.53 kWh
4,142A,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,1170.7,5,98.3,1404.85 kWh
7,142ES,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,685.0,4,71.9,821.97 kWh
8,142F,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,2598.2,23,47.4,3117.78 kWh
9,142K,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,449.2,3,62.9,539.08 kWh
12,144,Elektrik,HARAMİDERE,0.000,386.6,2,81.2,463.94 kWh
13,144A,Elektrik,İ.Ü. CERRAHPAŞA AVC. KAMP.,0.000,1167.5,6,81.7,1400.96 kWh
14,144B,Elektrik,KANUNİ SULTAN SÜLEYMAN,10.625,600.9,2,126.2,721.05 kWh
15,144H,Elektrik,KANUNİ SULTAN SÜLEYMAN,9.119,276.0,1,115.9,331.21 kWh
18,145,Elektrik,HARAMİDERE,2.941,2024.5,10,85.0,2429.43 kWh


In [39]:
# ── SONUÇLARI EXCEL OLARAK KAYDETME KODU ──

if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    # 1. HAT BAZLI DETAYLI ATAMA TABLOSUNU HAZIRLAMA
    hat_rows = []
    for h in H:
        for b in B:
            for y in Y:
                if J[h, b, y].X > 0.5:
                    tuketim = D[h] * V_E if y == "Elektrik" else D[h] * V_H
                    birim = "kWh" if y == "Elektrik" else "kg"
                    
                    hat_rows.append({
                        "Hat Kodu": h,
                        "Seçilen Yakıt": y,
                        "Atandığı İstasyon Peronu": b,
                        "İstasyona Erişim Mesafesi (km)": round(d[h, b], 3),
                        "Günlük Toplam Hat-km": round(D[h], 1),
                        "Araç Sayısı": A[h],
                        "Günlük Şarj/Dolum Yükü (dk)": round(U[h, y], 1),
                        "Günlük Yakıt Tüketimi": round(tuketim, 2),
                        "Tüketim Birimi": birim
                    })
    hat_df = pd.DataFrame(hat_rows).sort_values(["Seçilen Yakıt", "Hat Kodu"])

    # 2. İSTASYON BAZLI ALTYAPI VE DOLULUK TABLOSUNU HAZIRLAMA
    istasyon_rows = []
    for b in B:
        elec_open = X[b, "Elektrik"].X > 0.5
        hydro_open = X[b, "Hidrojen"].X > 0.5
        
        if elec_open or hydro_open:
            if elec_open and hydro_open:
                ist_type = "Hibrit (E+H)"
            else:
                ist_type = "Elektrik" if elec_open else "Hidrojen"
                
            atanan_hatlar = [h for h in H if (J[h, b, "Elektrik"].X > 0.5 or J[h, b, "Hidrojen"].X > 0.5)]
            hat_sayisi = len(atanan_hatlar)
            soket_sayisi = int(round(K[b, "Elektrik"].X)) if elec_open else 0
            pompa_sayisi = int(round(K[b, "Hidrojen"].X)) if hydro_open else 0
            
            if elec_open and soket_sayisi > 0:
                e_talep = sum(D[h] * V_E * J[h, b, "Elektrik"].X for h in H)
                e_doluluk = round((e_talep / (Cap_E * soket_sayisi)) * 100, 1)
                e_metrik = f"%{e_doluluk}"
            else:
                e_metrik = "-"
                
            if hydro_open and pompa_sayisi > 0:
                h_talep = sum(D[h] * V_H * J[h, b, "Hidrojen"].X for h in H)
                h_doluluk = round((h_talep / (Cap_H * pompa_sayisi)) * 100, 1)
                h_metrik = f"%{h_doluluk}"
            else:
                h_metrik = "-"
                
            istasyon_rows.append({
                "Açılan İstasyon Konumu": b,
                "Altyapı Türü": ist_type,
                "Kurulan Elk. Soket": soket_sayisi,
                "Kurulan H2 Pompası": pompa_sayisi,
                "Hizmet Verilen Hat Sayısı": hat_sayisi,
                "Elektrik Doluluk Oranı": e_metrik,
                "Hidrojen Doluluk Oranı": h_metrik,
                "Bağlı Hatlar (İlk 5)": ", ".join(atanan_hatlar[:5]) + ("..." if hat_sayisi > 5 else "")
            })
    istasyon_df = pd.DataFrame(istasyon_rows).sort_values("Hizmet Verilen Hat Sayısı", ascending=False)

    # 3. EXCEL DOSYASI OLARAK YAZDIRMA (İki ayrı sekme halinde)
    excel_dosya_adi = "iett_optimizasyon_sonuclari.xlsx"
    
    with pd.ExcelWriter(excel_dosya_adi, engine="openpyxl") as writer:
        hat_df.to_excel(writer, sheet_name="Hat Atamaları", index=False)
        istasyon_df.to_excel(writer, sheet_name="İstasyon Durumları", index=False)
        
    print(f"✔ Başarılı! Sonuçlar '{excel_dosya_adi}' adıyla kaydedildi.")
    print("Notebook'un çalıştığı klasörü kontrol ederek Excel dosyanı açabilirsin.")
else:
    print("Model henüz çözülmediği veya uygun çözüm bulunamadığı için Excel oluşturulamıyor.")

✔ Başarılı! Sonuçlar 'iett_optimizasyon_sonuclari.xlsx' adıyla kaydedildi.
Notebook'un çalıştığı klasörü kontrol ederek Excel dosyanı açabilirsin.


In [40]:
# ── AÇILAN İSTASYONLARIN KOORDİNATLARINI EXCEL'E KAYDETME KODU ──

if model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT]:
    istasyon_koordinat_rows = []
    
    for b in B:
        # İstasyonun Elektrik veya Hidrojen olarak açılıp açılmadığını kontrol edelim
        elec_open = X[b, "Elektrik"].X > 0.5
        hydro_open = X[b, "Hidrojen"].X > 0.5
        
        # Eğer istasyon aktifse koordinat bilgilerini ekleyelim
        if elec_open or hydro_open:
            # B_coord sözlüğünden enlem ve boylamı çekiyoruz
            enlem, boylam = B_coord.get(b, (None, None))
            
            # İstasyon Türünü Belirleme
            if elec_open and hydro_open:
                ist_turu = "Hibrit (E+H)"
            else:
                ist_turu = "Elektrik" if elec_open else "Hidrojen"
                
            istasyon_koordinat_rows.append({
                "İstasyon Noktası Adı": b,
                "Altyapı Türü": ist_turu,
                "Enlem (Latitude)": enlem,
                "Boylam (Longitude)": boylam
            })
            
    # DataFrame oluşturma
    istasyon_koordinat_df = pd.DataFrame(istasyon_koordinat_rows)
    
    # Excel dosyası olarak kaydetme
    excel_adi = "aktif_istasyon_koordinatlari.xlsx"
    istasyon_koordinat_df.to_excel(excel_adi, index=False, engine="openpyxl")
    
    print("="*60)
    print(f"✔ BAŞARILI! Toplam {len(istasyon_koordinat_df)} aktif istasyon listelendi.")
    print(f"✔ Dosya '{excel_adi}' adıyla başarıyla kaydedildi.")
    print("="*60)
    
    # Önizleme için ilk 10 satırı ekrana basalım
    display(istasyon_koordinat_df.head(10))
else:
    print("Model henüz çözülmediği veya uygun çözüm bulunamadığı için koordinat Excel'i oluşturulamıyor.")

✔ BAŞARILI! Toplam 30 aktif istasyon listelendi.
✔ Dosya 'aktif_istasyon_koordinatlari.xlsx' adıyla başarıyla kaydedildi.


,İstasyon Noktası Adı,Altyapı Türü,Enlem (Latitude),Boylam (Longitude)
0,KANUNİ SULTAN SÜLEYMAN,Hibrit (E+H),41.052274,28.762520
1,TÜYAP - METROBÜS,Elektrik,41.022490,28.623139
2,HACIOSMAN METRO,Hibrit (E+H),41.140057,29.030342
3,SARIYER,Elektrik,41.169050,29.057678
4,GİYİMKENT,Hidrojen,41.074212,28.871979
5,ARNAVUTKÖY PERONLAR,Elektrik,41.178340,28.750204
6,SULTANGAZİ PERONLAR,Hidrojen,41.104168,28.885177
7,MESCİD-İ SELAM,Hibrit (E+H),41.117617,28.851066
8,ÇATALCA,Hidrojen,41.146981,28.452449
9,İGTOT,Hidrojen,41.085612,28.722700


In [41]:
pip install folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [46]:
import pandas as pd
import folium
import math

# --- 1. Veriyi oku ---
df = pd.read_excel("aktif_istasyon_koordinatlari-harita.xlsx")

# Enlem/Boylam virgüllü ondalık olarak gelirse string -> float dönüşümü
for col in ["Enlem (Latitude)", "Boylam (Longitude)"]:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(",", ".").astype(float)

# --- 2. Harita merkezi ---
center_lat = df["Enlem (Latitude)"].mean()
center_lon = df["Boylam (Longitude)"].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=11, tiles="CartoDB voyager")

# --- 3. Sınıf -> ikon/renk eşlemesi ---
icon_map = {
    "Elektrik":     {"icon": "bolt",     "color": "orange", "circle": "#F1C40F"},  # sarı
    "Hidrojen":     {"icon": "tint",     "color": "blue",   "circle": "#2B8CBE"},  # mavi
    "Hibrit (E+H)": {"icon": "asterisk", "color": "green",  "circle": "#27AE60"},  # yeşil
}

# Kapasiteyi daire yarıçapına çeviren ölçek (alan ~ kapasite ile orantılı)
RADIUS_SCALE = 3.2  # büyütmek/küçültmek için bu sayıyı değiştir

# --- 4. Her tür için aç/kapa edilebilir katman (FeatureGroup) ---
groups = {}
for tur in df["Altyapı Türü"].unique():
    groups[tur] = folium.FeatureGroup(name=tur, show=True)

# --- 5. Marker'ları ilgili gruba ekle (3 katman: daire -> sayı -> ikon) ---
for _, row in df.iterrows():
    tur = row["Altyapı Türü"]
    style = icon_map.get(tur, {"icon": "info-sign", "color": "gray", "circle": "#95A5A6"})
    lat, lon = row["Enlem (Latitude)"], row["Boylam (Longitude)"]
    cap = row["Kapasite"]
    radius = math.sqrt(cap) * RADIUS_SCALE  # alan-orantılı, smooth ölçek
    fg = groups[tur]

    popup_html = f"""
    <b>{row['İstasyon Noktası Adı']}</b><br>
    Tür: {tur}<br>
    Kapasite: {cap}
    """

    # 5a. Alt katman: kapasite dairesi (yarı saydam -> smooth)
    folium.CircleMarker(
        location=[lat, lon],
        radius=radius,
        color=style["circle"],
        weight=1.5,
        fill=True,
        fill_color=style["circle"],
        fill_opacity=0.40,
        opacity=0.85,
    ).add_to(fg)

    # 5b. Orta katman: kapasite sayısı (dairenin merkezinde rozet)
    folium.Marker(
        location=[lat, lon],
        icon=folium.DivIcon(
            icon_size=(0, 0),
            icon_anchor=(0, 0),
            html=(
                '<div style="transform: translate(-50%, -50%);'
                'font-size:11px; font-weight:700; color:#222;'
                'text-shadow:0 0 3px #fff,0 0 3px #fff,0 0 3px #fff;'
                'white-space:nowrap;">'
                f'{cap}</div>'
            ),
        ),
    ).add_to(fg)

    # 5c. Üst katman: tür ikonu (pin) -> türle birlikte aç/kapanır
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=row["İstasyon Noktası Adı"],
        icon=folium.Icon(icon=style["icon"], prefix="fa", color=style["color"]),
    ).add_to(fg)

# Grupları haritaya ekle
for fg in groups.values():
    fg.add_to(m)

# Aç/kapa kontrolü (sağ üstte)
folium.LayerControl(collapsed=False).add_to(m)

# --- 6. Lejant ekle (basit HTML kutusu) ---
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background-color: white; padding: 10px 14px; border: 2px solid grey;
            border-radius: 6px; font-size: 14px;">
  <b>Legend</b><br>
  <i class="fa fa-bolt" style="color:orange"></i> Electric<br>
  <i class="fa fa-tint" style="color:blue"></i> Hydrogen<br>
  <i class="fa fa-asterisk" style="color:green"></i> Hybrid (E+H)<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# --- İkonları topluca gizleme kutusu (sadece daireler kalsın) ---
# .awesome-marker yalnızca pin ikonlarını hedefler; daireler (SVG) ve sayılar etkilenmez.
icon_toggle_html = """
<style>
  body.daire-modu .awesome-marker { display: none !important; }
</style>
<div style="position: fixed; top: 130px; right: 10px; z-index: 1000;
            background-color: white; padding: 8px 12px; border: 2px solid grey;
            border-radius: 6px; font-size: 13px;">
  <label style="cursor:pointer;">
    <input type="checkbox" checked
      onclick="document.body.classList.toggle('daire-modu', !this.checked)">
    İkonları göster
  </label>
</div>
"""
m.get_root().html.add_child(folium.Element(icon_toggle_html))

# --- 7. Kaydet ---
m.save("istasyon_haritasi.html")
print("Harita kaydedildi: istasyon_haritasi.html")

Harita kaydedildi: istasyon_haritasi.html
